<a href="https://colab.research.google.com/github/Lucianadeoliveira/gaTE-lab/blob/main/alphaFold/solForDiffEquations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import time

# 🌟 Parâmetros ajustáveis
params = {
    "k_VV1": 1,
    "k_NV1": 1,
    "k_V1V2": 1,
    "k_DV1": 1,
    "k_VV2": 1,
    "k_NV2": 1,
    "k_V2D": 1,
    "k_V2V3": 1,
    "k_V3D": 1,
    "k_FD": 1,
    "k_JD": 1,
    "k_DJ": 1,
    "k_JN1": 1,
    "k_N1N": 1,
    "k_DN1": 1,
}

# ⚙️ Sistema de EDOs
def ode_system(t, y, p):
    VEGFR1, VEGFR2, VEGFR3, DLL4, JAG1, NOTCH1, NICD, VEGF, FGF = y

    dVEGFR1_dt = (
        p["k_VV1"] * VEGF * VEGFR1 +
        p["k_NV1"] * NICD * VEGFR1 -
        p["k_V1V2"] * VEGFR1 * VEGFR2 -
        p["k_DV1"] * DLL4 * VEGFR1
    )

    dVEGFR2_dt = (
        p["k_VV2"] * VEGF * VEGFR2 -
        p["k_V1V2"] * VEGFR1 * VEGFR2 -
        p["k_NV2"] * NICD * VEGFR2 -
        p["k_V2D"] * VEGFR2 * DLL4 -
        p["k_V2V3"] * VEGFR2 * VEGFR3
    )

    dVEGFR3_dt = (
        p["k_V2V3"] * VEGFR2 * VEGFR3 -
        p["k_V3D"] * VEGFR3 * DLL4
    )

    dDLL4_dt = (
        p["k_FD"] * FGF * DLL4 +
        p["k_V2D"] * VEGFR2 * DLL4 +
        p["k_V3D"] * VEGFR3 * DLL4 -
        p["k_JD"] * JAG1 * DLL4 -
        p["k_DJ"] * DLL4 * JAG1 -
        p["k_DV1"] * DLL4 * VEGFR1 -
        p["k_DN1"] * DLL4 * NOTCH1
    )

    dJAG1_dt = (
        -p["k_JN1"] * JAG1 * NOTCH1 -
        p["k_V3D"] * VEGFR3 * DLL4 -
        p["k_DJ"] * DLL4 * JAG1 -
        p["k_JD"] * JAG1 * DLL4
    )

    dNOTCH1_dt = (
        p["k_JN1"] * JAG1 * NOTCH1 -
        p["k_N1N"] * NOTCH1 * NICD +
        p["k_DN1"] * DLL4 * NOTCH1
    )

    dNICD_dt = (
        p["k_N1N"] * NOTCH1 * NICD -
        p["k_NV1"] * NICD * VEGFR1 -
        p["k_NV2"] * NICD * VEGFR2
    )

    dVEGF_dt = 0
    dFGF_dt = 0

    return [
        dVEGFR1_dt, dVEGFR2_dt, dVEGFR3_dt, dDLL4_dt, dJAG1_dt,
        dNOTCH1_dt, dNICD_dt, dVEGF_dt, dFGF_dt
    ]

# 🧷 Condições iniciais
y0 = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 5.0, 5.0]

# 🕓 Intervalo de tempo
t_span = (0, 100)
t_eval = np.linspace(*t_span, 500)

# ⏱️ Início da simulação
print("🚀 Iniciando integração...")
start_time = time.time()

sol = solve_ivp(
    fun=lambda t, y: ode_system(t, y, params),
    t_span=t_span,
    y0=y0,
    t_eval=t_eval,
    method='RK45',
    vectorized=False
)

# ⏱️ Tempo final
end_time = time.time()
duration = end_time - start_time

# ✅ Checagem
if sol.success:
    print(f"✅ Integração concluída em {duration:.2f} segundos.")
else:
    print(f"❌ Erro na integração: {sol.message}")
    raise RuntimeError("A integração falhou. Verifique o modelo e as condições.")

# 📊 Plot
plt.figure(figsize=(12, 6))
labels = ["VEGFR1", "VEGFR2", "VEGFR3", "DLL4", "JAG1", "NOTCH1", "NICD", "VEGF", "FGF"]
for i in range(len(labels)):
    plt.plot(sol.t, sol.y[i], label=labels[i])

plt.xlabel("Time")
plt.ylabel("Concentration")
plt.title("Dynamics of VEGF/FGF signaling network")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


🚀 Iniciando integração...
